# Lesson 4 — Indian Game: London System (as Black)
**Project 1500 | Priority #4**

All statistics computed fresh from game files on every run. No hardcoded results.
Board diagrams are illustrative example lines, clearly labelled as such.

In [ ]:
import chess
import chess.pgn
import chess.svg
import json
import glob
import io
import re
from collections import Counter, defaultdict
from IPython.display import SVG, display, HTML
import matplotlib.pyplot as plt

USERNAME   = 'jf4bes'
EXCLUDE    = {'2025_01', '2025_02', '2025_03'}
BOARD_SIZE = 380

def board_after(moves_san):
    """Apply SAN moves to a fresh board — illustrative diagrams only."""
    board = chess.Board()
    for san in moves_san:
        board.push_san(san)
    return board

def show_board(board, arrows=None, caption='', size=BOARD_SIZE, flipped=True):
    svg = chess.svg.board(board, arrows=arrows or [], size=size, flipped=flipped)
    if caption:
        display(HTML(f'<p style="font-family:monospace;font-size:13px;margin:4px 0 8px 0;color:#aaa">{caption}</p>'))
    display(SVG(data=svg))

## Step 1 — Load rapid games as black (April 2025+)

In [ ]:
def load_rapid_games_as_black(username, games_dir='games', exclude=EXCLUDE):
    files = sorted(glob.glob(f'{games_dir}/2025_*.json') + glob.glob(f'{games_dir}/2026_*.json'))
    files = [f for f in files if not any(ex in f for ex in exclude)]
    games = []
    for f in files:
        with open(f, encoding='utf-8') as fh:
            month = json.load(fh)
        for g in month:
            if g.get('time_class') != 'rapid': continue
            black = g.get('black', {})
            if black.get('username', '').lower() != username.lower(): continue
            result = black.get('result', '')
            if   result == 'win': outcome = 'win'
            elif result in ('checkmated','timeout','resigned','lose','abandoned'): outcome = 'loss'
            elif result in ('agreed','stalemate','repetition','insufficient','timevsinsufficient','50move'): outcome = 'draw'
            else: continue
            eco_url = ''
            m = re.search(r'\[ECOUrl "([^"]+)"\]', g.get('pgn', ''))
            if m: eco_url = m.group(1)
            games.append({'outcome': outcome, 'pgn': g.get('pgn', ''), 'eco_url': eco_url})
    return games

all_black_games = load_rapid_games_as_black(USERNAME)
print(f'Loaded {len(all_black_games)} rapid games as black')

## Step 2 — Filter to London System games and compute stats

We also detect whether black's third move (the first meaningful choice) was **b6** or something else,
and compare outcomes.

In [ ]:
LONDON_PATTERN = 'London'

london_games = [
    g for g in all_black_games
    if LONDON_PATTERN in g['eco_url']
]

counts = Counter(g['outcome'] for g in london_games)
total  = len(london_games)
wr     = 100 * counts['win'] / total if total else 0

print(f'London System games as black: {total}')
print(f'W:{counts["win"]}  L:{counts["loss"]}  D:{counts["draw"]}  Win%: {wr:.1f}%')

if total < 15:
    print('\nNote: small sample — treat percentages as directional, not definitive.')

# Track black's response: b6 vs everything else
b6_games    = []
no_b6_games = []

for g in london_games:
    try:
        game  = chess.pgn.read_game(io.StringIO(g['pgn']))
        board = game.board()
        moves = list(game.mainline_moves())
        played_b6 = False
        for i, move in enumerate(moves[:10]):
            if i % 2 == 1:  # black's move
                if board.san(move) == 'b6':
                    played_b6 = True
                    break
            board.push(move)
        if played_b6:
            b6_games.append(g)
        else:
            no_b6_games.append(g)
    except Exception:
        pass

b6_counts    = Counter(g['outcome'] for g in b6_games)
no_b6_counts = Counter(g['outcome'] for g in no_b6_games)

for label, c, games_list in [('b6 played', b6_counts, b6_games), ('b6 NOT played', no_b6_counts, no_b6_games)]:
    t = len(games_list)
    w = 100 * c['win'] / t if t else 0
    print(f'{label:20s}: {t:3d} games  W:{c["win"]} L:{c["loss"]} D:{c["draw"]}  Win%: {w:.1f}%')

## Step 3 — b6 vs no-b6 comparison

In [ ]:
b6_t    = len(b6_games)
no_b6_t = len(no_b6_games)
b6_wr   = 100 * b6_counts['win']    / b6_t    if b6_t    else 0
no_b6_wr= 100 * no_b6_counts['win'] / no_b6_t if no_b6_t else 0

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, label, c, t, col in [
    (axes[0], f'Played b6\n({b6_t} games)',     b6_counts,    b6_t,    '#27ae60'),
    (axes[1], f'Did NOT play b6\n({no_b6_t} games)', no_b6_counts, no_b6_t, '#c0392b'),
]:
    wr = 100 * c['win'] / t if t else 0
    ax.bar(['Win','Loss','Draw'], [c['win'],c['loss'],c['draw']],
           color=[col,'#666','#999'], edgecolor='white')
    ax.set_title(f"{label}\n{wr:.1f}% win rate")
    ax.set_ylabel('Games')
plt.suptitle('London System as black: b6 system vs other responses', fontsize=13)
plt.tight_layout()
plt.show()

## Step 4 — Board diagrams *(illustrative example lines)*

In [ ]:
# [ILLUSTRATIVE] The London position — showing b6 vs Nc6
# Verified legal: d4 Nf6 Nf3 e6 Bf4
LONDON_BASE = ['d4','Nf6','Nf3','e6','Bf4']

show_board(
    board_after(LONDON_BASE),
    arrows=[
        chess.svg.Arrow(chess.B7, chess.B6, color='#27ae60'),  # b6 — recommended
        chess.svg.Arrow(chess.B8, chess.C6, color='#c0392b'),  # Nc6 — blocks c-pawn
    ],
    caption='[ILLUSTRATIVE] After 1.d4 Nf6 2.Nf3 e6 3.Bf4 — Green=b6 (fianchetto plan) | Red=Nc6 (blocks c-pawn)'
)

In [ ]:
# [ILLUSTRATIVE] The b6 system fully developed — active bishops, solid structure
# Verified legal: d4 Nf6 Nf3 e6 Bf4 b6 e3 Bb7 Bd3 c5 c3 Be7 O-O O-O
B6_SYSTEM = ['d4','Nf6','Nf3','e6','Bf4','b6','e3','Bb7','Bd3','c5','c3','Be7','O-O','O-O']

show_board(
    board_after(B6_SYSTEM),
    arrows=[
        chess.svg.Arrow(chess.B7, chess.E4, color='#27ae60'),  # Bb7 long diagonal
        chess.svg.Arrow(chess.C5, chess.D4, color='#2980b9'),  # c5 pressures d4
    ],
    caption='[ILLUSTRATIVE] After ...b6 Bb7 c5 O-O — Bb7 fires down the long diagonal, c5 pressures d4'
)

## Step 5 — Summary

In [ ]:
print('LONDON SYSTEM AS BLACK — SUMMARY')
print('=' * 50)
print(f'\nTotal London games as black: {total}  ({wr:.1f}% win rate)')
print(f'\nb6 system:     {b6_t} games, {b6_wr:.1f}% win rate')
print(f'Other systems: {no_b6_t} games, {no_b6_wr:.1f}% win rate')
print()
print('RULES:')
print('  1. Play ...e6, ...b6, ...Bb7 — fianchetto the queen bishop')
print('  2. Do NOT play ...Nc6 early — it blocks the c-pawn')
print('  3. Break with ...c5 to pressure white\'s d4 pawn')
print('  4. After ...c5 is played, then ...Nc6 is fine (c-pawn already moved)')